In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-2b"

# -------------------------------
# 🔹 LOAD MODEL
# -------------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [2]:
# -------------------------------
# 🔹 BASIC INFO
# -------------------------------
print("\n===== MODEL STRUCTURE =====")
print(model)

print("\n===== CONFIG =====")
print(model.config)

# -------------------------------
# 🔹 PARAMETER SHAPES
# -------------------------------
print("\n===== PARAMETER SHAPES =====")
for name, param in model.named_parameters():
    print(f"{name:60s} {tuple(param.shape)}")

# -------------------------------
# 🔹 SELECT LAYER
# -------------------------------
layer_id = 0
layer = model.model.layers[layer_id]

print(f"\n===== INSPECTING LAYER {layer_id} =====")

# -------------------------------
# 🔹 WEIGHTS
# -------------------------------
print("\n-- Attention Weights --")
print("Q:", layer.self_attn.q_proj.weight.shape)
print("K:", layer.self_attn.k_proj.weight.shape)
print("V:", layer.self_attn.v_proj.weight.shape)
print("O:", layer.self_attn.o_proj.weight.shape)

print("\n-- MLP Weights --")
print("Gate:", layer.mlp.gate_proj.weight.shape)
print("Up:", layer.mlp.up_proj.weight.shape)
print("Down:", layer.mlp.down_proj.weight.shape)

print("\n-- Norm Layers --")
print("Input Norm:", layer.input_layernorm.weight.shape)
print("Post-Attn Norm:", layer.post_attention_layernorm.weight.shape)

# -------------------------------
# 🔹 ACTIVATION HOOKS (ROBUST)
# -------------------------------
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        # Handle tuple outputs safely
        if isinstance(output, tuple):
            activations[name] = output[0]
        else:
            activations[name] = output
    return hook

# Register hooks
layer.self_attn.register_forward_hook(hook_fn("attention_output"))
layer.mlp.register_forward_hook(hook_fn("mlp_output"))

# Also hook Q/K/V projections (VERY DEEP)
layer.self_attn.q_proj.register_forward_hook(hook_fn("Q"))
layer.self_attn.k_proj.register_forward_hook(hook_fn("K"))
layer.self_attn.v_proj.register_forward_hook(hook_fn("V"))

# -------------------------------
# 🔹 INPUT
# -------------------------------
text = "The cat sat on the mat"
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# -------------------------------
# 🔹 FORWARD PASS
# -------------------------------
with torch.no_grad():
    outputs = model(**inputs)

# -------------------------------
# 🔹 ACTIVATION ANALYSIS
# -------------------------------
print("\n===== ACTIVATIONS =====")

for name, act in activations.items():
    print(f"\n{name}:")
    print(f"  shape = {act.shape}")
    print(f"  mean  = {act.mean().item():.4f}")
    print(f"  std   = {act.std().item():.4f}")
    print(f"  min   = {act.min().item():.4f}")
    print(f"  max   = {act.max().item():.4f}")

# -------------------------------
# 🔹 LAYER-BY-LAYER FLOW
# -------------------------------
hidden_states = outputs.hidden_states

print("\n===== LAYER-WISE OUTPUT =====")
for i, h in enumerate(hidden_states):
    print(f"Layer {i:2d}: shape={h.shape}, mean={h.mean().item():.4f}, std={h.std().item():.4f}")

# -------------------------------
# 🔹 TOKEN REPRESENTATION
# -------------------------------
print("\n===== TOKEN REPRESENTATION =====")

last_layer = hidden_states[-1]  # [batch, seq, hidden]
token_vector = last_layer[0, -1]

print("Last token vector shape:", token_vector.shape)
print("First 10 values:", token_vector[:10])

# -------------------------------
# 🔹 LM HEAD (PREDICTION)
# -------------------------------
print("\n===== LM HEAD =====")

lm_head = model.lm_head
print("LM Head weight shape:", lm_head.weight.shape)

logits = lm_head(last_layer)
print("Logits shape:", logits.shape)

probs = torch.softmax(logits[0, -1], dim=-1)
topk = torch.topk(probs, k=5)

print("\nTop 5 predicted tokens:")
for i in range(5):
    token_id = topk.indices[i].item()
    token = tokenizer.decode([token_id])
    prob = topk.values[i].item()
    print(f"{token} → {prob:.4f}")

# -------------------------------
# 🔹 MANUAL MLP COMPUTATION
# -------------------------------
print("\n===== MANUAL MLP COMPUTATION =====")

mlp = layer.mlp
x = hidden_states[layer_id][0]  # one sequence

gate = mlp.gate_proj(x)
up = mlp.up_proj(x)

activated = torch.nn.functional.gelu(gate)
combined = activated * up

down = mlp.down_proj(combined)

print("Gate shape:", gate.shape)
print("Up shape:", up.shape)
print("After GELU:", activated.shape)
print("After gating:", combined.shape)
print("Final output:", down.shape)


print(model.__class__)


===== MODEL STRUCTURE =====
GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)


In [11]:
text = "The cat sat on the mat"
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# -------------------------------
# 🔵 PREFILL
# -------------------------------
with torch.no_grad():
    outputs = model(**inputs, use_cache=True)

past_key_values = outputs.past_key_values

print("\n===== AFTER PREFILL =====")

for i, layer in enumerate(past_key_values):
    k, v = layer[:2]
    print(f"Layer {i} K shape:", k.shape)
    print(f"Layer {i} V shape:", v.shape)
# -------------------------------
# 🔴 DECODE LOOP
# -------------------------------
next_token = inputs["input_ids"][:, -1:]

num_steps = 5

for step in range(num_steps):

    print(f"\n===== DECODE STEP {step+1} =====")

    with torch.no_grad():
        outputs = model(
            input_ids=next_token,
            past_key_values=past_key_values,
            use_cache=True
        )

    past_key_values = outputs.past_key_values
    logits = outputs.logits

    # Get next token
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

    print("Generated token:", tokenizer.decode(next_token[0]))

# -------------------------------
# 🔍 INSPECT K/V (Layer 0)
# -------------------------------
layer0 = next(iter(past_key_values))
k, v = layer0[:2]

print("Layer 0 K shape:", k.shape)
print("Layer 0 V shape:", v.shape)


===== AFTER PREFILL =====
Layer 0 K shape: torch.Size([1, 1, 7, 256])
Layer 0 V shape: torch.Size([1, 1, 7, 256])
Layer 1 K shape: torch.Size([1, 1, 7, 256])
Layer 1 V shape: torch.Size([1, 1, 7, 256])
Layer 2 K shape: torch.Size([1, 1, 7, 256])
Layer 2 V shape: torch.Size([1, 1, 7, 256])
Layer 3 K shape: torch.Size([1, 1, 7, 256])
Layer 3 V shape: torch.Size([1, 1, 7, 256])
Layer 4 K shape: torch.Size([1, 1, 7, 256])
Layer 4 V shape: torch.Size([1, 1, 7, 256])
Layer 5 K shape: torch.Size([1, 1, 7, 256])
Layer 5 V shape: torch.Size([1, 1, 7, 256])
Layer 6 K shape: torch.Size([1, 1, 7, 256])
Layer 6 V shape: torch.Size([1, 1, 7, 256])
Layer 7 K shape: torch.Size([1, 1, 7, 256])
Layer 7 V shape: torch.Size([1, 1, 7, 256])
Layer 8 K shape: torch.Size([1, 1, 7, 256])
Layer 8 V shape: torch.Size([1, 1, 7, 256])
Layer 9 K shape: torch.Size([1, 1, 7, 256])
Layer 9 V shape: torch.Size([1, 1, 7, 256])
Layer 10 K shape: torch.Size([1, 1, 7, 256])
Layer 10 V shape: torch.Size([1, 1, 7, 256])
Lay

In [12]:
print("\n===== STEP 0: EMBEDDING =====")
print("Embedding output shape:", x.shape)  # [1, seq_len, 2048]

# -------------------------------
# 🔹 SELECT LAYER
# -------------------------------
layer = model.model.layers[0]

# -------------------------------
# 🔹 STEP 1: INPUT LAYERNORM
# -------------------------------
x_norm = layer.input_layernorm(x)

print("\n===== STEP 1: INPUT NORM =====")
print("After RMSNorm:", x_norm.shape)

# -------------------------------
# 🔹 STEP 2: Q, K, V PROJECTIONS
# -------------------------------
Q = layer.self_attn.q_proj(x_norm)
K = layer.self_attn.k_proj(x_norm)
V = layer.self_attn.v_proj(x_norm)

print("\n===== STEP 2: Q, K, V =====")
print("Q shape:", Q.shape)  # [1, seq, 2048]
print("K shape:", K.shape)  # [1, seq, 256]
print("V shape:", V.shape)  # [1, seq, 256]

# -------------------------------
# 🔹 STEP 3: RESHAPE INTO HEADS
# -------------------------------
num_heads = model.config.num_attention_heads
head_dim = Q.shape[-1] // num_heads

Q_heads = Q.view(1, -1, num_heads, head_dim).transpose(1, 2)
K_heads = K.view(1, -1, 1, head_dim).transpose(1, 2)  # MQA → 1 KV head
V_heads = V.view(1, -1, 1, head_dim).transpose(1, 2)

print("\n===== STEP 3: HEAD SPLIT =====")
print("Q_heads:", Q_heads.shape)  # [batch, heads, seq, head_dim]
print("K_heads:", K_heads.shape)
print("V_heads:", V_heads.shape)

# -------------------------------
# 🔹 STEP 4: ATTENTION SCORES
# -------------------------------
attn_scores = torch.matmul(Q_heads, K_heads.transpose(-1, -2))

print("\n===== STEP 4: ATTENTION SCORES =====")
print("Q × Kᵀ shape:", attn_scores.shape)  # [batch, heads, seq, seq]

# -------------------------------
# 🔹 STEP 5: SOFTMAX
# -------------------------------
attn_probs = torch.softmax(attn_scores, dim=-1)

print("\n===== STEP 5: SOFTMAX =====")
print("Attention probs shape:", attn_probs.shape)

# -------------------------------
# 🔹 STEP 6: APPLY TO V
# -------------------------------
attn_output = torch.matmul(attn_probs, V_heads)

print("\n===== STEP 6: ATTENTION OUTPUT =====")
print("Shape after multiplying with V:", attn_output.shape)

# -------------------------------
# 🔹 STEP 7: MERGE HEADS
# -------------------------------
attn_output = attn_output.transpose(1, 2).contiguous()
attn_output = attn_output.view(1, -1, num_heads * head_dim)

print("\n===== STEP 7: MERGE HEADS =====")
print("Merged shape:", attn_output.shape)

# -------------------------------
# 🔹 STEP 8: OUTPUT PROJECTION
# -------------------------------
attn_output = layer.self_attn.o_proj(attn_output)

print("\n===== STEP 8: OUTPUT PROJECTION =====")
print("After o_proj:", attn_output.shape)

# -------------------------------
# 🔹 STEP 9: RESIDUAL ADD
# -------------------------------
x = x + attn_output

print("\n===== STEP 9: RESIDUAL =====")
print("After residual:", x.shape)

# -------------------------------
# 🔹 STEP 10: POST ATTENTION NORM
# -------------------------------
x_norm2 = layer.post_attention_layernorm(x)

print("\n===== STEP 10: POST NORM =====")
print("After norm:", x_norm2.shape)

# -------------------------------
# 🔹 STEP 11: MLP
# -------------------------------
gate = layer.mlp.gate_proj(x_norm2)
up = layer.mlp.up_proj(x_norm2)

print("\n===== STEP 11: MLP EXPANSION =====")
print("Gate shape:", gate.shape)  # [1, seq, 16384]
print("Up shape:", up.shape)

# Activation + gating
activated = torch.nn.functional.gelu(gate)
combined = activated * up

print("\n===== STEP 12: GATING =====")
print("After GELU:", activated.shape)
print("After multiply:", combined.shape)

# Down projection
mlp_output = layer.mlp.down_proj(combined)

print("\n===== STEP 13: MLP DOWN =====")
print("MLP output shape:", mlp_output.shape)

# -------------------------------
# 🔹 STEP 14: FINAL RESIDUAL
# -------------------------------
x = x + mlp_output

print("\n===== STEP 14: FINAL OUTPUT =====")
print("Layer output shape:", x.shape)


===== STEP 0: EMBEDDING =====
Embedding output shape: torch.Size([7, 2048])

===== STEP 1: INPUT NORM =====
After RMSNorm: torch.Size([7, 2048])

===== STEP 2: Q, K, V =====
Q shape: torch.Size([7, 2048])
K shape: torch.Size([7, 256])
V shape: torch.Size([7, 256])

===== STEP 3: HEAD SPLIT =====
Q_heads: torch.Size([1, 8, 7, 256])
K_heads: torch.Size([1, 1, 7, 256])
V_heads: torch.Size([1, 1, 7, 256])

===== STEP 4: ATTENTION SCORES =====
Q × Kᵀ shape: torch.Size([1, 8, 7, 7])

===== STEP 5: SOFTMAX =====
Attention probs shape: torch.Size([1, 8, 7, 7])

===== STEP 6: ATTENTION OUTPUT =====
Shape after multiplying with V: torch.Size([1, 8, 7, 256])

===== STEP 7: MERGE HEADS =====
Merged shape: torch.Size([1, 7, 2048])

===== STEP 8: OUTPUT PROJECTION =====
After o_proj: torch.Size([1, 7, 2048])

===== STEP 9: RESIDUAL =====
After residual: torch.Size([1, 7, 2048])

===== STEP 10: POST NORM =====
After norm: torch.Size([1, 7, 2048])

===== STEP 11: MLP EXPANSION =====
Gate shape: torch